# 06 — Shortcomings, measured
Each item: what the code does, one measurement, one consequence. Nothing is fixed here; this is the list the next version is engineered against.

In [1]:
import os, sys, json, math, collections, re, hashlib
import numpy as np
sys.path.insert(0, os.path.abspath("../tools"))
import ptol_bin, monad_py as mp
ROOT  = os.path.abspath("..")
STORE = os.path.join(ROOT, "dirty", "monad_wordnet_v4_20260519.bin")
CBIN  = os.path.join(ROOT, "build", "PtolC_eb28ce8", "ptolemy")
SPACES = "[\u200b\u2009\u2008\u2007\u2006\u2005\u2004\u2003]"   # the eight Fermat separators

In [2]:
m = mp.Monad.from_bin(STORE)
from nltk.corpus import wordnet as wn
lem = sorted({l.lower() for s in wn.all_synsets() for l in s.lemma_names() if l.isalpha() and l.isascii()})
lem = [w for w in lem if mp.token_accept(w)]
print(len(lem), "WordNet single-token ASCII lemmas pass the prose filter")

76719 WordNet single-token ASCII lemmas pass the prose filter


## 1. Address resolution collapses with word length
See notebook 02. 8+ letters → slot 0.

In [3]:
print("lemmas at slot 0:", sum(1 for w in lem if mp.word_coords(w)[0]==0), "of", len(lem))
print("distinct slots reached by all lemmas:", len({mp.word_coords(w)[0] for w in lem}), "of 25000")

lemmas at slot 0: 49018 of 76719


distinct slots reached by all lemmas: 11599 of 25000


## 2. The displayed word is the highest-`E` arrival in the slot, so most displayed words are not words anyone meant

In [4]:
by = collections.defaultdict(list)
for w in lem: by[mp.word_coords(w)[0]].append(w)
inbin = sum(1 for w in lem if w in m.wm)
print("WordNet lemmas that are their slot's displayed word:", inbin, "of", len(lem), f"({100*inbin/len(lem):.1f} %)")
print("mean lemmas per occupied slot (excluding slot 0):", round(np.mean([len(v) for i,v in by.items() if i]),2))

WordNet lemmas that are their slot's displayed word: 3919 of 76719 (5.1 %)
mean lemmas per occupied slot (excluding slot 0): 2.39


Consequence: `is`→`nanti`, `name`→`emite`, `tree`→`typhon`, `water`→`defrg`. The *weights* (β, A) live on the slot and are shared by every word that addresses there; only the label is arbitrary.

## 3. The zeros are approximations
Notebook 02: error up to 1.8 at n = 21, 0.28 at n = 25 000.

## 4. Fixed reply length
`actual_max = 4 + int(50·ΣT/(ΣT+ΣV))` with ΣV ≈ 5.7·10³ → 4 for short prompts.

In [5]:
beta = np.array(m.beta); E = np.array([m.vocab[i]["E"] if i in m.vocab else 0 for i in range(25000)])
print("sum_V (all occupied slots):", float(np.sum(np.where(E>0, beta*E*E, 0))))

sum_V (all occupied slots): 5668.27671247942


## 5. β saturates
β is capped at 7.552 and most occupied slots reach the cap, so β no longer discriminates between slots.

In [6]:
occ = np.array(sorted(m.vocab))
print("slots at the cap:", int((beta[occ] >= 7.55).sum()), "of", len(occ), f"({100*(beta[occ]>=7.55).mean():.1f} %)")

slots at the cap: 1826 of 24485 (7.5 %)


## 6. The tokenizer discards every non-ASCII byte
Greek, Chinese, Arabic, Hebrew, Coptic, accented Latin letters all become separators. `ağaç` → `a`, `a` (both dropped, 1 char). A multilingual test through this engine must use ASCII orthography.

## 7. Documentation and code disagree (one case)
The comment before the demotic selection says "first output word chosen from first 200 qualifying walk candidates"; the code scans all N slots. The code is what ran.

## 8. The word map is not persisted
`state_save` writes vocab entries only, so after a reload `monad_wm_get` knows only the displayed words; every other word is `[new]` and its `E` is recomputed. Same slot, same `E` (the
address is deterministic), so the reply is unchanged — but the `[new]` flag in the trace is not evidence about whether the word was ever seen.

## 9. Open: the separator
The screenshot reply has no visible spaces (zero-width separators) while the same binary on a pipe prints a visible U+2003 em space in one prompt of the standard eight. Which band appears depends on `J/J_max`.
Whether a tty changes anything was not tested.

## 10. `prose_seen` belongs to the slot, not to the displayed word
`monad_learn_ex` sets `prose_seen` (0 none, 1 prose, 2 WordNet text, 3 both) on the *slot* each time any word addresses there. A slot flagged 3 ("verified common") may display a word that was never in WordNet, because a different word that shares the slot was.

In [7]:
ps = collections.Counter(v["ps"] for v in m.vocab.values())
print("prose_seen over occupied slots:", dict(ps))
flag3_not_wn = [v["word"] for v in m.vocab.values() if v["ps"]==3 and not wn.synsets(v["word"])]
print(len(flag3_not_wn), "slots flagged 3 display a word with no WordNet synset, e.g.", flag3_not_wn[:12])

prose_seen over occupied slots: {3: 13594, 1: 10538, 2: 158, 0: 195}
8968 slots flagged 3 display a word with no WordNet synset, e.g. ['seemy', 'carre', 'azv', 'diem', 'uxp', 'arcos', 'ofkind', 'sary', 'tpass', 'utmcc', 'minv', 'jaune']
